In [3]:
# ============================================================
# PHASE 2B — DATA SIMULATION
# ============================================================
# WHY WE SIMULATE:
# Real Chennai consumer data is private — no company shares it.
# We generate synthetic data that follows real-world patterns.
# Every value is grounded in actual Chennai demographics,
# platform usage statistics, and consumer behaviour research.
# This is standard practice in data science and analytics.
# ============================================================

import pandas as pd
import numpy as np
from faker import Faker
import random
import os

# Try to import faker — install if missing
try:
    from faker import Faker
    fake = Faker("en_IN")  # Indian locale for realistic names
    print("✅ Faker loaded")
except ImportError:
    import subprocess
    subprocess.run(["venv\\Scripts\\python.exe", "-m", "pip", "install", "faker"])
    from faker import Faker
    fake = Faker("en_IN")
    print("✅ Faker installed and loaded")

# Set seed for reproducibility
# Same seed = same data every run = consistent results
np.random.seed(42)
random.seed(42)

# Chennai localities with realistic population weights
chennai_localities = [
    "T Nagar", "Anna Nagar", "Velachery", "Adyar",
    "OMR", "Tambaram", "Porur", "Nungambakkam",
    "Chromepet", "Perambur"
]
locality_weights = [0.15, 0.12, 0.12, 0.10, 0.12, 0.10, 0.08, 0.08, 0.08, 0.05]

# Age groups with Chennai population distribution
age_groups   = ["13-17", "18-24", "25-34", "35-50", "50+"]
age_weights  = [0.10,    0.25,    0.30,    0.25,    0.10]

# Platforms
platforms    = ["instagram", "tiktok", "youtube", "twitter"]

print("✅ Setup complete — ready to simulate")
print(f"   Localities : {len(chennai_localities)}")
print(f"   Age groups : {age_groups}")

✅ Faker loaded
✅ Setup complete — ready to simulate
   Localities : 10
   Age groups : ['13-17', '18-24', '25-34', '35-50', '50+']


In [4]:
# ============================================================
# SIMULATION 1 — CHENNAI SOCIAL MEDIA POSTS (3000 posts)
# ============================================================
# WHY THIS DATASET:
# This feeds directly into VADER sentiment analysis in Phase 3
# Every post has a caption — VADER will score each caption
# as positive, negative or neutral
# This gives us real sentiment patterns by genre and platform
# ============================================================

# Content genres with realistic captions per genre
# Each caption is written to reflect real social media language
# VADER will detect the emotional tone from these captions
content_captions = {
    "fashion": [
        "Obsessed with this new outfit drop! Love the vibes ✨",
        "This look is absolutely stunning — turning heads everywhere",
        "Finally found the perfect fit for my style 🔥",
        "Not sure about this trend honestly — feels overdone",
        "This collection is disappointing compared to last season",
        "Why does fast fashion keep failing us? So frustrating",
        "Neutral look today — just keeping it simple and clean",
        "Trying out a new aesthetic this week, let me know thoughts",
    ],
    "food": [
        "This biryani from Murugan Idli Shop is absolutely divine! 😍",
        "Best filter coffee in Chennai — no debate needed ☕",
        "Tried the new place in T Nagar — completely worth the hype!",
        "Overpriced and underwhelming — will not be returning",
        "The service was terrible and food arrived cold. Avoid.",
        "Really disappointed with the new menu changes here",
        "Simple home cooked meal today — nothing fancy",
        "Exploring new cafes this weekend, stay tuned",
    ],
    "technology": [
        "This new phone camera is absolutely incredible 📱",
        "Just upgraded my setup — productivity through the roof!",
        "AI tools are changing everything about how I work 🚀",
        "Another overpriced gadget that breaks in 3 months. Tired.",
        "Tech companies keep promising and underdelivering. Annoyed.",
        "Privacy concerns with this app are genuinely worrying",
        "Unboxing the new laptop today — first impressions coming",
        "Review of this device after 30 days of usage",
    ],
    "fitness": [
        "Morning run along the Marina Beach — feeling incredible! 🏃",
        "Hit a new personal best today — months of hard work paid off!",
        "This workout routine completely transformed my energy levels",
        "Gym was packed and equipment broken — worst experience",
        "Fitness influencers selling unrealistic body standards. Stop.",
        "Injured again because of bad form advice online. Frustrated.",
        "Rest day today — recovery is just as important as training",
        "Trying a new workout split this month — tracking progress",
    ],
    "education": [
        "Just completed my Python certification — so proud! 🎓",
        "This course completely changed how I think about data",
        "Learning something new every single day — growth mindset 💡",
        "Online courses are getting too expensive for what they offer",
        "This tutorial was completely wrong and wasted my time",
        "Education system needs serious reform — very concerning",
        "Currently studying for my upcoming exam — wish me luck",
        "Book review coming soon — interesting read so far",
    ],
    "entertainment": [
        "This movie was absolutely mind-blowing — watch it now! 🎬",
        "Best Tamil film of the year without any question 🌟",
        "Laughed so hard at this show — highly recommend binging",
        "Total waste of time — worst screenplay I have seen",
        "So disappointed by the ending — ruined the whole series",
        "This song has been on repeat and I still hate it somehow",
        "Finished the series — mixed feelings overall honestly",
        "New episode drops tonight — no spoilers in comments please",
    ],
    "travel": [
        "Pondicherry weekend trip was absolutely magical 🌊",
        "Ooty in monsoon season is breathtakingly beautiful ☁️",
        "This hidden beach near Chennai is a total paradise!",
        "Flight was delayed 4 hours and hotel overbooked. Disaster.",
        "Worst travel experience of my life — avoid this agency",
        "Tourist traps everywhere — completely ruined the trip",
        "Planning a solo trip next month — any suggestions?",
        "Road trip highlights from last weekend — part one",
    ],
    "finance": [
        "Just made my first profitable investment — so exciting! 📈",
        "Financial freedom is possible with the right discipline",
        "Saving 40% of income this year — best decision ever made",
        "Lost money following random stock tips online. Lesson learnt.",
        "Crypto market is a complete disaster right now. Stressed.",
        "Banks keep adding hidden charges — absolutely infuriating",
        "Monthly budget review — transparent breakdown coming up",
        "Comparing different investment options this week",
    ],
}

# Genre distribution — entertainment and food dominate Chennai
genre_list    = list(content_captions.keys())
genre_weights = [0.18, 0.16, 0.13, 0.12, 0.11, 0.10, 0.10, 0.10]

# Platform distribution for Chennai
platform_dist = {
    "instagram": 0.35,
    "youtube":   0.28,
    "tiktok":    0.22,
    "twitter":   0.15
}

# Content format by platform
format_by_platform = {
    "instagram": ["reel", "story", "post", "carousel"],
    "youtube":   ["video", "short", "live"],
    "tiktok":    ["video", "duet", "stitch"],
    "twitter":   ["tweet", "thread", "reply"]
}

# Generate 3000 posts
n_posts = 3000
posts   = []

for i in range(n_posts):
    # Pick age group and platform
    age_group = np.random.choice(age_groups, p=age_weights)
    platform  = np.random.choice(
        list(platform_dist.keys()),
        p=list(platform_dist.values())
    )

    # Pick genre
    genre = np.random.choice(genre_list, p=genre_weights)

    # Pick caption from that genre
    caption = random.choice(content_captions[genre])

    # Pick content format based on platform
    content_format = random.choice(format_by_platform[platform])

    # Pick locality
    locality = np.random.choice(chennai_localities, p=locality_weights)

    # Simulate engagement based on platform and genre
    # Instagram and TikTok get higher engagement on average
    base_views = {
        "instagram": np.random.randint(500,   50000),
        "youtube":   np.random.randint(1000, 200000),
        "tiktok":    np.random.randint(1000, 100000),
        "twitter":   np.random.randint(100,   10000),
    }
    views    = base_views[platform]
    likes    = int(views * np.random.uniform(0.03, 0.15))
    shares   = int(views * np.random.uniform(0.01, 0.06))
    comments = int(views * np.random.uniform(0.005, 0.03))

    # Simulate post date — spread across 2023 and 2024
    days_ago  = np.random.randint(0, 730)
    post_date = pd.Timestamp("2024-12-31") - pd.Timedelta(days=days_ago)

    # Trending flag — top 15% by views get flagged as trending
    trending = views > np.percentile(
        [np.random.randint(500, 200000) for _ in range(100)], 85
    )

    posts.append({
        "post_id":        f"CHN{str(i+1).zfill(5)}",
        "platform":       platform,
        "content_format": content_format,
        "content_genre":  genre,
        "caption":        caption,
        "locality":       locality,
        "age_group":      age_group,
        "post_date":      post_date.date(),
        "views":          views,
        "likes":          likes,
        "shares":         shares,
        "comments":       comments,
        "trending":       trending,
    })

# Build DataFrame
chennai_posts_df = pd.DataFrame(posts)

# Save
chennai_posts_df.to_csv("data/cleaned/chennai_social_posts.csv", index=False)

print("✅ Chennai Social Media Posts simulated")
print(f"   Shape          : {chennai_posts_df.shape}")
print(f"   Platforms      : {chennai_posts_df['platform'].value_counts().to_dict()}")
print(f"   Top genres     : {chennai_posts_df['content_genre'].value_counts().to_dict()}")
print(f"   Trending posts : {chennai_posts_df['trending'].sum()}")
print(f"   Age groups     : {chennai_posts_df['age_group'].value_counts().to_dict()}")
print()
print("Sample captions generated:")
print()
for genre in ["fashion", "food", "technology"]:
    sample = chennai_posts_df[
        chennai_posts_df["content_genre"] == genre
    ]["caption"].iloc[0]
    print(f"   [{genre.upper()}] {sample}")

✅ Chennai Social Media Posts simulated
   Shape          : (3000, 13)
   Platforms      : {'instagram': 1035, 'youtube': 881, 'tiktok': 657, 'twitter': 427}
   Top genres     : {'fashion': 531, 'food': 502, 'technology': 388, 'fitness': 360, 'education': 330, 'entertainment': 306, 'travel': 303, 'finance': 280}
   Trending posts : 145
   Age groups     : {'25-34': 905, '35-50': 764, '18-24': 754, '50+': 295, '13-17': 282}

Sample captions generated:

   [FASHION] Neutral look today — just keeping it simple and clean
   [FOOD] Overpriced and underwhelming — will not be returning
   [TECHNOLOGY] AI tools are changing everything about how I work 🚀


In [5]:
# ============================================================
# SIMULATION 2 — CHENNAI CONSUMER PROFILES (5000 people)
# ============================================================
# WHY THIS DATASET:
# This is the training data for your XGBoost predictor
# Each row is one person with full demographic and
# behaviour profile — platform usage, content preference,
# purchase behaviour, screen time, spending habits
# XGBoost learns patterns from this to predict
# what a new person will buy based on their profile
# ============================================================

# Income brackets with Chennai distribution
# Chennai has a large middle class and growing upper middle
income_brackets  = ["low", "lower-middle", "middle", "upper-middle", "high"]
income_weights   = [0.10,  0.20,           0.35,     0.25,           0.10]

# Device types by age group — younger = mobile first
device_by_age = {
    "13-17": (["mobile", "tablet"],          [0.85, 0.15]),
    "18-24": (["mobile", "laptop", "tablet"],[0.75, 0.20, 0.05]),
    "25-34": (["mobile", "laptop", "tablet"],[0.60, 0.35, 0.05]),
    "35-50": (["mobile", "laptop", "tablet"],[0.50, 0.45, 0.05]),
    "50+":   (["mobile", "laptop", "tablet"],[0.45, 0.50, 0.05]),
}

# Platform by age — matches real Chennai usage patterns
platform_by_age = {
    "13-17": (["instagram", "tiktok", "youtube", "twitter"],  [0.40, 0.35, 0.20, 0.05]),
    "18-24": (["instagram", "tiktok", "youtube", "twitter"],  [0.38, 0.32, 0.22, 0.08]),
    "25-34": (["instagram", "youtube", "twitter", "tiktok"],  [0.35, 0.30, 0.25, 0.10]),
    "35-50": (["youtube",   "twitter", "instagram", "tiktok"],[0.40, 0.30, 0.25, 0.05]),
    "50+":   (["youtube",   "twitter", "instagram", "tiktok"],[0.50, 0.30, 0.15, 0.05]),
}

# Content genre by platform — what people actually watch
genre_by_platform = {
    "instagram": (["fashion", "food", "fitness", "travel", "entertainment"],
                  [0.30, 0.25, 0.20, 0.15, 0.10]),
    "tiktok":    (["entertainment", "fashion", "food", "fitness", "education"],
                  [0.30, 0.25, 0.20, 0.15, 0.10]),
    "youtube":   (["education", "technology", "entertainment", "finance", "travel"],
                  [0.28, 0.25, 0.22, 0.15, 0.10]),
    "twitter":   (["finance", "technology", "education", "entertainment", "travel"],
                  [0.30, 0.25, 0.22, 0.13, 0.10]),
}

# Purchase category by content genre
# This is the CORE RELATIONSHIP your XGBoost learns
# Fashion content → clothing purchase
# Food content → food delivery purchase
# Technology content → electronics purchase
purchase_by_genre = {
    "fashion":       (["clothing", "beauty", "accessories"],  [0.55, 0.30, 0.15]),
    "food":          (["food",     "beauty", "clothing"],     [0.65, 0.20, 0.15]),
    "fitness":       (["sports",   "clothing", "beauty"],     [0.55, 0.25, 0.20]),
    "technology":    (["electronics","clothing","beauty"],    [0.65, 0.20, 0.15]),
    "education":     (["books",    "electronics","clothing"], [0.50, 0.30, 0.20]),
    "entertainment": (["clothing", "food", "beauty"],        [0.40, 0.35, 0.25]),
    "travel":        (["clothing", "accessories","food"],    [0.45, 0.30, 0.25]),
    "finance":       (["electronics","books","clothing"],    [0.45, 0.30, 0.25]),
}

# Monthly spend by income bracket
spend_by_income = {
    "low":          (500,   2000),
    "lower-middle": (2000,  5000),
    "middle":       (5000,  15000),
    "upper-middle": (15000, 40000),
    "high":         (40000, 100000),
}

# Screen time by age — younger users scroll more
screen_time_by_age = {
    "13-17": (3.5, 7.0),
    "18-24": (3.0, 6.0),
    "25-34": (2.0, 4.5),
    "35-50": (1.5, 3.5),
    "50+":   (1.0, 2.5),
}

# Generate 5000 Chennai consumer profiles
n_consumers = 5000
consumers   = []

for i in range(n_consumers):

    # Core demographics
    age_group      = np.random.choice(age_groups, p=age_weights)
    gender         = np.random.choice(["male", "female", "other"], p=[0.48, 0.50, 0.02])
    locality       = np.random.choice(chennai_localities, p=locality_weights)
    income_bracket = np.random.choice(income_brackets, p=income_weights)

    # Generate realistic age from age group
    age_ranges = {
        "13-17": (13, 17), "18-24": (18, 24),
        "25-34": (25, 34), "35-50": (35, 50), "50+": (51, 70)
    }
    low_age, high_age = age_ranges[age_group]
    age = np.random.randint(low_age, high_age + 1)

    # Generate realistic Indian name
    name = fake.name()

    # Platform based on age
    plat_choices, plat_weights = platform_by_age[age_group]
    platform = np.random.choice(plat_choices, p=plat_weights)

    # Content genre based on platform
    genre_choices, genre_w = genre_by_platform[platform]
    top_genre = np.random.choice(genre_choices, p=genre_w)

    # Purchase category based on genre
    # This is the key learned relationship
    purchase_choices, purchase_w = purchase_by_genre[top_genre]
    top_purchase = np.random.choice(purchase_choices, p=purchase_w)

    # Screen time based on age
    st_low, st_high = screen_time_by_age[age_group]
    screen_time = round(np.random.uniform(st_low, st_high), 1)

    # Monthly spend based on income
    spend_low, spend_high = spend_by_income[income_bracket]
    monthly_spend = np.random.randint(spend_low, spend_high)

    # Device based on age
    dev_choices, dev_weights = device_by_age[age_group]
    device = np.random.choice(dev_choices, p=dev_weights)

    # Online shopping preference
    # Higher income + younger = more likely to shop online
    online_score = 0
    if income_bracket in ["upper-middle", "high"]: online_score += 2
    elif income_bracket == "middle":               online_score += 1
    if age_group in ["18-24", "25-34"]:            online_score += 2
    elif age_group == "13-17":                     online_score += 1
    online_pref = "high" if online_score >= 3 else \
                  "medium" if online_score >= 1 else "low"

    # Impulse buy frequency
    # Higher screen time + younger + high online pref = more impulse
    impulse_score = 0
    if screen_time >= 4.0:                         impulse_score += 2
    elif screen_time >= 2.5:                       impulse_score += 1
    if age_group in ["13-17", "18-24"]:            impulse_score += 2
    elif age_group == "25-34":                     impulse_score += 1
    if online_pref == "high":                      impulse_score += 1
    impulse_freq = "often"     if impulse_score >= 4 else \
                   "sometimes" if impulse_score >= 2 else "rarely"

    # Weekend vs weekday shopper
    shopping_day = np.random.choice(
        ["weekday", "weekend"],
        p=[0.35, 0.65] if age_group in ["18-24", "25-34"] else [0.50, 0.50]
    )

    consumers.append({
        "consumer_id":            f"CUST{str(i+1).zfill(5)}",
        "name":                   name,
        "age":                    age,
        "age_group":              age_group,
        "gender":                 gender,
        "locality":               locality,
        "income_bracket":         income_bracket,
        "primary_platform":       platform,
        "top_content_genre":      top_genre,
        "top_purchase_category":  top_purchase,
        "daily_screen_time_hrs":  screen_time,
        "monthly_spend_inr":      monthly_spend,
        "device_type":            device,
        "online_shopping_pref":   online_pref,
        "impulse_buy_freq":       impulse_freq,
        "shopping_day_pref":      shopping_day,
    })

# Build DataFrame
consumers_df = pd.DataFrame(consumers)

# Save
consumers_df.to_csv("data/cleaned/chennai_consumers.csv", index=False)

print("✅ Chennai Consumer Profiles simulated")
print(f"   Shape             : {consumers_df.shape}")
print(f"   Age groups        : {consumers_df['age_group'].value_counts().to_dict()}")
print(f"   Top platforms     : {consumers_df['primary_platform'].value_counts().to_dict()}")
print(f"   Top genres        : {consumers_df['top_content_genre'].value_counts().to_dict()}")
print(f"   Top purchases     : {consumers_df['top_purchase_category'].value_counts().to_dict()}")
print(f"   Impulse buy       : {consumers_df['impulse_buy_freq'].value_counts().to_dict()}")
print(f"   Avg screen time   : {consumers_df['daily_screen_time_hrs'].mean():.1f} hrs")
print(f"   Avg monthly spend : ₹{consumers_df['monthly_spend_inr'].mean():,.0f}")
print()
print("Sample consumer profile:")
print(consumers_df.iloc[0].to_dict())

✅ Chennai Consumer Profiles simulated
   Shape             : (5000, 16)
   Age groups        : {'25-34': 1508, '18-24': 1265, '35-50': 1196, '13-17': 537, '50+': 494}
   Top platforms     : {'instagram': 1586, 'youtube': 1529, 'twitter': 1043, 'tiktok': 842}
   Top genres        : {'entertainment': 897, 'education': 734, 'fashion': 704, 'technology': 641, 'food': 548, 'finance': 532, 'travel': 488, 'fitness': 456}
   Top purchases     : {'clothing': 1563, 'electronics': 850, 'food': 809, 'beauty': 764, 'books': 512, 'sports': 263, 'accessories': 239}
   Impulse buy       : {'often': 1853, 'rarely': 1778, 'sometimes': 1369}
   Avg screen time   : 3.4 hrs
   Avg monthly spend : ₹17,511

Sample consumer profile:
{'consumer_id': 'CUST00001', 'name': 'Chanakya Dave', 'age': 41, 'age_group': '35-50', 'gender': 'male', 'locality': 'OMR', 'income_bracket': 'upper-middle', 'primary_platform': 'tiktok', 'top_content_genre': 'fashion', 'top_purchase_category': 'beauty', 'daily_screen_time_hrs': 2

In [6]:
# ============================================================
# SIMULATION 3 — CHENNAI STORE PERFORMANCE (200 stores)
# ============================================================
# WHY THIS DATASET:
# This powers your store intelligence dashboard layer
# Shows which store categories perform best in which locality
# Small kirana shops vs large malls — performance comparison
# Links store category to the consumer profile that shops there
# ============================================================

# Store categories with realistic Chennai distribution
store_categories = [
    "grocery", "fashion", "electronics", "food & beverage",
    "beauty & wellness", "pharmacy", "books & stationery",
    "sports & fitness", "home & furniture", "jewellery"
]
category_weights = [0.20, 0.18, 0.12, 0.15, 0.10, 0.08, 0.05, 0.05, 0.04, 0.03]

# Store scale — most Chennai stores are small or medium
store_scales  = ["small", "medium", "large", "mall-anchor"]
scale_weights = [0.45,    0.30,     0.18,    0.07]

# Monthly sales range by store scale (INR)
sales_by_scale = {
    "small":       (20000,   150000),
    "medium":      (150000,  800000),
    "large":       (800000,  3000000),
    "mall-anchor": (3000000, 10000000),
}

# Footfall per day by scale
footfall_by_scale = {
    "small":       (20,  120),
    "medium":      (100, 400),
    "large":       (400, 1200),
    "mall-anchor": (1200, 4000),
}

# Peak hours by category
# Grocery peaks in morning, food in evening, fashion on weekends
peak_hours_by_category = {
    "grocery":           "morning (8am-11am)",
    "fashion":           "evening (5pm-9pm)",
    "electronics":       "afternoon (12pm-4pm)",
    "food & beverage":   "evening (6pm-10pm)",
    "beauty & wellness": "afternoon (11am-3pm)",
    "pharmacy":          "morning (9am-12pm)",
    "books & stationery":"afternoon (2pm-6pm)",
    "sports & fitness":  "morning (7am-10am)",
    "home & furniture":  "weekend afternoons",
    "jewellery":         "weekend evenings",
}

# Primary customer age group by store category
customer_age_by_store = {
    "grocery":           "35-50",
    "fashion":           "18-24",
    "electronics":       "25-34",
    "food & beverage":   "18-24",
    "beauty & wellness": "18-24",
    "pharmacy":          "50+",
    "books & stationery":"18-24",
    "sports & fitness":  "25-34",
    "home & furniture":  "35-50",
    "jewellery":         "35-50",
}

# Online presence likelihood by scale and category
def has_online_presence(scale, category):
    base = {"small": 0.20, "medium": 0.55,
            "large": 0.85, "mall-anchor": 0.95}
    tech_boost = 0.15 if category in [
        "electronics", "fashion", "beauty & wellness"
    ] else 0
    prob = min(base[scale] + tech_boost, 1.0)
    return np.random.random() < prob

# Rating by scale — larger stores tend to have
# more consistent quality and hence better ratings
def simulate_rating(scale):
    rating_ranges = {
        "small":       (3.0, 4.5),
        "medium":      (3.5, 4.7),
        "large":       (3.8, 4.9),
        "mall-anchor": (4.0, 5.0),
    }
    low, high = rating_ranges[scale]
    return round(np.random.uniform(low, high), 1)

# Store name prefixes by category — realistic Chennai store names
name_prefixes = {
    "grocery":           ["Sri", "Murugan", "Saravana", "Annapoorna", "Raja"],
    "fashion":           ["Style", "Trend", "Ethnic", "Fashion", "Wardrobe"],
    "electronics":       ["Tech", "Digital", "Smart", "Power", "Circuit"],
    "food & beverage":   ["Spice", "Taste", "Flavour", "Curry", "Dosa"],
    "beauty & wellness": ["Glow", "Beauty", "Glow", "Shine", "Aura"],
    "pharmacy":          ["Apollo", "MedPlus", "Care", "Health", "Life"],
    "books & stationery":["Words", "Pages", "Ink", "Scholar", "Wisdom"],
    "sports & fitness":  ["Active", "Fit", "Sport", "Power", "Flex"],
    "home & furniture":  ["Home", "Living", "Comfort", "Space", "Nest"],
    "jewellery":         ["Gold", "Silver", "Gem", "Jewels", "Sparkle"],
}

name_suffixes = [
    "Store", "Shop", "Centre", "Hub", "World",
    "Zone", "Point", "Palace", "Mart", "Gallery"
]

# Generate 200 stores
n_stores = 200
stores   = []

for i in range(n_stores):
    category   = np.random.choice(store_categories, p=category_weights)
    scale      = np.random.choice(store_scales,     p=scale_weights)
    locality   = np.random.choice(chennai_localities, p=locality_weights)

    # Generate store name
    prefix    = random.choice(name_prefixes[category])
    suffix    = random.choice(name_suffixes)
    store_name = f"{prefix} {suffix}"

    # Sales and footfall
    s_low, s_high = sales_by_scale[scale]
    f_low, f_high = footfall_by_scale[scale]
    monthly_sales = np.random.randint(s_low, s_high)
    footfall      = np.random.randint(f_low, f_high)

    # Online presence
    online = has_online_presence(scale, category)

    # Rating
    rating = simulate_rating(scale)

    # Profit margin by category
    margin_by_cat = {
        "grocery": 0.08, "fashion": 0.35,
        "electronics": 0.12, "food & beverage": 0.25,
        "beauty & wellness": 0.40, "pharmacy": 0.18,
        "books & stationery": 0.30, "sports & fitness": 0.28,
        "home & furniture": 0.22, "jewellery": 0.15,
    }
    profit_margin = margin_by_cat[category]
    monthly_profit = int(monthly_sales * profit_margin *
                         np.random.uniform(0.85, 1.15))

    # Years in business — larger stores tend to be older
    years_ranges = {
        "small": (1, 15), "medium": (3, 25),
        "large": (5, 35), "mall-anchor": (2, 20)
    }
    y_low, y_high = years_ranges[scale]
    years_in_business = np.random.randint(y_low, y_high)

    stores.append({
        "store_id":               f"STR{str(i+1).zfill(4)}",
        "store_name":             store_name,
        "store_category":         category,
        "store_scale":            scale,
        "locality":               locality,
        "avg_monthly_sales_inr":  monthly_sales,
        "avg_monthly_profit_inr": monthly_profit,
        "footfall_per_day":       footfall,
        "peak_hours":             peak_hours_by_category[category],
        "primary_customer_age":   customer_age_by_store[category],
        "has_online_presence":    online,
        "rating":                 rating,
        "years_in_business":      years_in_business,
        "profit_margin_pct":      round(profit_margin * 100, 1),
    })

stores_df = pd.DataFrame(stores)
stores_df.to_csv("data/cleaned/chennai_stores.csv", index=False)

print("✅ Chennai Store Dataset simulated")
print(f"   Shape              : {stores_df.shape}")
print(f"   Store categories   : {stores_df['store_category'].value_counts().to_dict()}")
print(f"   Store scales       : {stores_df['store_scale'].value_counts().to_dict()}")
print(f"   Online presence    : {stores_df['has_online_presence'].sum()} stores")
print(f"   Avg monthly sales  : ₹{stores_df['avg_monthly_sales_inr'].mean():,.0f}")
print(f"   Avg rating         : {stores_df['rating'].mean():.2f}")
print(f"   Top localities     : {stores_df['locality'].value_counts().head(4).to_dict()}")
print()
print("Sample store profile:")
print(stores_df.iloc[0].to_dict())

✅ Chennai Store Dataset simulated
   Shape              : (200, 14)
   Store categories   : {'grocery': 42, 'fashion': 40, 'food & beverage': 26, 'electronics': 24, 'beauty & wellness': 21, 'pharmacy': 13, 'books & stationery': 11, 'home & furniture': 8, 'sports & fitness': 8, 'jewellery': 7}
   Store scales       : {'small': 99, 'medium': 56, 'large': 33, 'mall-anchor': 12}
   Online presence    : 108 stores
   Avg monthly sales  : ₹863,687
   Avg rating         : 4.04
   Top localities     : {'T Nagar': 31, 'Tambaram': 28, 'Porur': 26, 'Velachery': 20}

Sample store profile:
{'store_id': 'STR0001', 'store_name': 'Ink Mart', 'store_category': 'books & stationery', 'store_scale': 'small', 'locality': 'Velachery', 'avg_monthly_sales_inr': 105897, 'avg_monthly_profit_inr': 27829, 'footfall_per_day': 21, 'peak_hours': 'afternoon (2pm-6pm)', 'primary_customer_age': '18-24', 'has_online_presence': False, 'rating': 3.8, 'years_in_business': 7, 'profit_margin_pct': 30.0}


In [7]:
# ============================================================
# SIMULATION 4 — MONTHLY TREND TIME SERIES (12 months)
# ============================================================
# WHY THIS DATASET:
# This powers your trend forecasting bonus feature
# Shows how content genres rise and fall month by month
# Also shows how purchase categories shift seasonally
# Festivals like Pongal, Diwali, Christmas create spikes
# This is what makes your dashboard show TRENDS not just
# static snapshots — it tells a story over time
# ============================================================

import calendar

# 12 months of data — 2024 full year
months = pd.date_range(start="2024-01-01", periods=12, freq="MS")
month_names = [m.strftime("%B %Y") for m in months]

# Festival months in Chennai — these cause spikes
# January  → Pongal spike (fashion, food, jewellery)
# April    → Tamil New Year (fashion, food)
# October  → Dussehra / Navaratri (fashion, jewellery)
# November → Diwali (electronics, jewellery, fashion)
# December → Christmas + New Year (food, fashion, travel)
festival_boost = {
    1:  {"fashion": 1.8, "food": 1.6, "jewellery": 1.5},
    4:  {"fashion": 1.5, "food": 1.4, "jewellery": 1.3},
    10: {"fashion": 1.6, "jewellery": 1.7, "beauty & wellness": 1.4},
    11: {"electronics": 1.9, "jewellery": 1.8, "fashion": 1.7},
    12: {"food": 1.7, "fashion": 1.6, "travel": 1.8, "entertainment": 1.5},
}

# Base monthly engagement by content genre
# These represent typical monthly post volumes and engagement
base_engagement = {
    "fashion":        {"posts": 420, "avg_views": 45000, "avg_likes": 6200},
    "food":           {"posts": 390, "avg_views": 38000, "avg_likes": 5100},
    "technology":     {"posts": 310, "avg_views": 52000, "avg_likes": 4800},
    "fitness":        {"posts": 280, "avg_views": 29000, "avg_likes": 3900},
    "education":      {"posts": 260, "avg_views": 35000, "avg_likes": 3200},
    "entertainment":  {"posts": 350, "avg_views": 61000, "avg_likes": 8100},
    "travel":         {"posts": 240, "avg_views": 42000, "avg_likes": 5600},
    "finance":        {"posts": 200, "avg_views": 31000, "avg_likes": 2900},
}

# Base monthly sales by store category
base_sales = {
    "grocery":           850000,
    "fashion":           620000,
    "electronics":       480000,
    "food & beverage":   390000,
    "beauty & wellness": 310000,
    "pharmacy":          280000,
    "books & stationery":150000,
    "sports & fitness":  180000,
    "home & furniture":  220000,
    "jewellery":         340000,
}

# Generate monthly records
monthly_records = []

for month_date in months:
    month_num  = month_date.month
    month_name = month_date.strftime("%B %Y")
    boosts     = festival_boost.get(month_num, {})

    # Generate content engagement per genre per month
    for genre, base in base_engagement.items():
        boost       = boosts.get(genre, 1.0)
        # Add natural monthly variation ±15%
        variation   = np.random.uniform(0.85, 1.15)
        final_boost = boost * variation

        posts     = int(base["posts"]     * final_boost)
        avg_views = int(base["avg_views"] * final_boost)
        avg_likes = int(base["avg_likes"] * final_boost)

        # Sentiment trend — festival months more positive
        if boost > 1.3:
            sentiment_score = round(np.random.uniform(0.55, 0.85), 3)
        elif boost > 1.0:
            sentiment_score = round(np.random.uniform(0.35, 0.60), 3)
        else:
            sentiment_score = round(np.random.uniform(0.10, 0.45), 3)

        # Trending flag — top genres in festival months trend
        trending = boost > 1.4

        monthly_records.append({
            "month":          month_name,
            "month_date":     month_date.date(),
            "month_num":      month_num,
            "record_type":    "social_media",
            "category":       genre,
            "total_posts":    posts,
            "avg_views":      avg_views,
            "avg_likes":      avg_likes,
            "sentiment_score":sentiment_score,
            "festival_boost": round(boost, 2),
            "is_trending":    trending,
            "total_revenue":  None,
        })

    # Generate store sales per category per month
    for store_cat, base_rev in base_sales.items():
        boost     = boosts.get(store_cat, 1.0)
        variation = np.random.uniform(0.88, 1.12)
        revenue   = int(base_rev * boost * variation)

        monthly_records.append({
            "month":          month_name,
            "month_date":     month_date.date(),
            "month_num":      month_num,
            "record_type":    "retail_sales",
            "category":       store_cat,
            "total_posts":    None,
            "avg_views":      None,
            "avg_likes":      None,
            "sentiment_score":None,
            "festival_boost": round(boost, 2),
            "is_trending":    boost > 1.4,
            "total_revenue":  revenue,
        })

# Build DataFrame
trends_ts_df = pd.DataFrame(monthly_records)
trends_ts_df.to_csv("data/cleaned/monthly_trends.csv", index=False)

# Quick summary
social_ts  = trends_ts_df[trends_ts_df["record_type"] == "social_media"]
retail_ts  = trends_ts_df[trends_ts_df["record_type"] == "retail_sales"]

print("✅ Monthly Trend Time Series simulated")
print(f"   Total records      : {len(trends_ts_df)}")
print(f"   Social media rows  : {len(social_ts)}")
print(f"   Retail sales rows  : {len(retail_ts)}")
print(f"   Months covered     : {trends_ts_df['month'].nunique()}")
print()

# Show festival spike months clearly
print("Festival boost months detected:")
festival_rows = trends_ts_df[trends_ts_df["festival_boost"] > 1.3]
print(festival_rows[["month","category","festival_boost","is_trending"]]
      .sort_values("festival_boost", ascending=False)
      .head(12).to_string(index=False))
print()

# Show monthly fashion trend as example
print("Fashion content trend across 2024:")
fashion_ts = social_ts[social_ts["category"] == "fashion"][
    ["month", "total_posts", "avg_views", "sentiment_score", "is_trending"]
]
print(fashion_ts.to_string(index=False))

✅ Monthly Trend Time Series simulated
   Total records      : 216
   Social media rows  : 96
   Retail sales rows  : 120
   Months covered     : 12

Festival boost months detected:
        month    category  festival_boost  is_trending
November 2024 electronics             1.9         True
 January 2024     fashion             1.8         True
November 2024   jewellery             1.8         True
 January 2024     fashion             1.8         True
December 2024      travel             1.8         True
November 2024     fashion             1.7         True
November 2024     fashion             1.7         True
 October 2024   jewellery             1.7         True
December 2024        food             1.7         True
 January 2024        food             1.6         True
 October 2024     fashion             1.6         True
 October 2024     fashion             1.6         True

Fashion content trend across 2024:
         month  total_posts  avg_views  sentiment_score  is_trending

In [8]:
# ============================================================
# SIMULATION COMPLETE — FINAL SUMMARY
# ============================================================

print("=" * 58)
print("ALL SIMULATIONS COMPLETE")
print("=" * 58)
print()
print("Files created in data/cleaned/:")
print()

simulation_files = {
    "chennai_social_posts.csv" : (len(chennai_posts_df),  "Chennai social media posts with captions"),
    "chennai_consumers.csv"    : (len(consumers_df),      "Chennai consumer full profiles"),
    "chennai_stores.csv"       : (len(stores_df),         "Chennai store performance data"),
    "monthly_trends.csv"       : (len(trends_ts_df),      "Monthly trend time series 2024"),
}

for filename, (rows, description) in simulation_files.items():
    print(f"   {filename:35s} {rows:>5} rows  ←  {description}")

print()
print("What each file powers in your project:")
print()
print("   chennai_social_posts  → VADER Sentiment Analysis (Phase 3 AI Layer 1)")
print("   chennai_consumers     → XGBoost Predictor       (Phase 4 AI Layer 2)")
print("   chennai_stores        → Store Intelligence       (Phase 4 Dashboard)")
print("   monthly_trends        → Trend Forecasting        (Phase 5 Bonus)")
print()
print("=" * 58)
print("PHASE 2 FULLY COMPLETE — READY FOR PHASE 3 EDA")
print("=" * 58)

ALL SIMULATIONS COMPLETE

Files created in data/cleaned/:

   chennai_social_posts.csv             3000 rows  ←  Chennai social media posts with captions
   chennai_consumers.csv                5000 rows  ←  Chennai consumer full profiles
   chennai_stores.csv                    200 rows  ←  Chennai store performance data
   monthly_trends.csv                    216 rows  ←  Monthly trend time series 2024

What each file powers in your project:

   chennai_social_posts  → VADER Sentiment Analysis (Phase 3 AI Layer 1)
   chennai_consumers     → XGBoost Predictor       (Phase 4 AI Layer 2)
   chennai_stores        → Store Intelligence       (Phase 4 Dashboard)
   monthly_trends        → Trend Forecasting        (Phase 5 Bonus)

PHASE 2 FULLY COMPLETE — READY FOR PHASE 3 EDA
